In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# Load PDF
loader = PyPDFLoader("../data/classic_rock_facts.pdf")
docs = loader.load()

In [ ]:
# Chunk it
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
all_splits = text_splitter.split_documents(docs)

In [ ]:
# Creaet the vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = InMemoryVectorStore(embeddings)

_ = vector_store.add_documents(all_splits)

In [21]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information from the PDF."""
    retrieved_docs = vector_store.similarity_search(query, k=3)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs



prompt = (
    "You have access to a tool that retrieves context from a PDF knowledge base. "
    "Use it to answer user questions as accurately as possible."
)

agent = create_agent(
    'gpt-4o-mini',
    tools = [retrieve_context], 
    system_prompt=prompt
)


In [23]:
response = agent.invoke(
    {"messages": ["What is this pdf about?"]}
    )

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

The PDF titled "rock_curiosities" contains various interesting facts and trivia related to classic rock music. It covers a range of topics, including anecdotes about famous musicians, song inspirations, and notable events in rock history. For example, it mentions that Axl Rose is an anagram for oral sex, discusses the implications of Billy Joel's lyrics, and details the background behind songs by notable artists like The Cure and Queen. The document consists of three pages filled with diverse rock-related curiosities.
